<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/cmaes_practical_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 致谢

本 Notebook 最初整理用于日本船舶海洋工学会第 354 回 KFR Seminar“面向船舶海洋工程师的黑盒优化研讨会”。原作者特别感谢大阪大学牧敦生教授提供整理这份实践指南的机会，也感谢宫内新喜博士在 CMA-ES 船舶海洋工程应用方面的合作与经验分享。很多工程经验只有在真实应用中才会暴露出来，本教程正是希望把这些经验系统化。

# 目的

很多应用研究者都会遇到类似问题：**“CMA-ES 已经跑起来了，但为什么得不到理想解？”** 本指南不重复推导 CMA-ES 的全部理论，而是从实际使用角度总结如何设置算法、如何从实验结果反推问题性质，以及什么时候应该重新考虑问题定式化。

# 参考资料

本指南与作者在 GECCO 长期讲授的 CMA-ES 教程互补。GECCO 教程更关注算法设计原理和各组件作用，本 Notebook 更关注实际怎么用、怎么诊断。

Youhei Akimoto and Nikolaus Hansen. 2022. *CMA-ES and advanced adaptation mechanisms*. GECCO '22 Companion, 1243–1268. https://doi.org/10.1145/3520304.3533648

教程视频：https://www.youtube.com/watch?v=7VBKLH3oDuw

本仓库 `0`–`6` 号 Notebook 已经按原理顺序给出中文基础教程。

# 推荐实现

原教程使用 DD-CMA-ES：Y. Akimoto and N. Hansen, *Diagonal Acceleration for Covariance Matrix Adaptation Evolution Strategies*, Evolutionary Computation 28(3), 2020。

参考代码：https://gist.github.com/youheiakimoto/1180b67b5a0b1265c204cba991fa8518

完整算法机制与精简可运行实现见本仓库 `6_advanced_adaptation_mechanisms.ipynb`。本实践指南不再重复粘贴数百行相同实现，而把重点放在如何解释日志和选择配置。

# DD-CMA-ES 的参数应该怎么看

DD-CMA-ES 从多元正态分布生成多个候选解，评估并排序，再根据排名更新分布。其搜索分布写成
$$\mathcal{N}(m,\sigma^2DCD).$$

* $m$（日志常写 `xmean`）：均值向量，即当前搜索中心。
* $\sigma$（`sigma`）：全局步长，控制整个分布的总体尺度。
* $D$：对角尺度矩阵，学习不同设计变量的相对敏感度。
* $C$：相关矩阵，学习变量之间的耦合方向。
* `S`：$C$ 特征值的平方根，可以理解为相关结构各主轴的相对尺度。

因此只看最佳目标函数值远远不够；`sigma`、`D` 和 `S` 往往能直接告诉你问题为什么难、搜索是否真的还在进行。

## 示例：Ellipsoid-Cigar

考虑 $d$ 维 Ellipsoid-Cigar：
$$f(x)=10^4\sum_{i=1}^{d}\left(10^{3\frac{i-1}{d-1}}x_i\right)^2+\frac{1-10^4}{d}\left(\sum_{i=1}^{d}10^{3\frac{i-1}{d-1}}x_i\right)^2.$$
不同变量尺度差异很大，同时还存在一个由 $v=(1/\sqrt d,\dots,1/\sqrt d)$ 定义的低敏感相关方向。其 Hessian 可写为
$$\nabla^2f=2D_{ell}(10^4I+(1-10^4)vv^T)D_{ell},$$
逆矩阵为
$$(\nabla^2f)^{-1}=\frac12D_{ell}^{-1}(10^{-4}I+vv^T)D_{ell}^{-1}.$$
这类问题非常适合观察 `D` 如何学习变量尺度，以及 `C/S` 如何学习非坐标轴方向的相关结构。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def ellcig(X):
    X = np.atleast_2d(X)
    d = np.logspace(0, 3, num=X.shape[1])
    v = np.ones(X.shape[1]) / np.sqrt(X.shape[1])
    Y = X * d
    return 1e4 * np.sum(Y**2, axis=1) + (1.0 - 1e4) * np.dot(Y, v)**2

# 如何读取 CMA-ES 的优化结果

## 1. 先看最佳目标值，但不要停在这里
`fmin` 是每一代候选解中的最小目标函数值。如果它已经达到工程要求，当然最重要。但当 `fmin` 暂时不再下降时，不能立刻判断“算法收敛了”或“算法失败了”。CMA-ES 可能正在用很多代学习协方差结构。

## 2. 再看 `sigma`：搜索是否真的收缩
如果 $\sigma^{(t)}/\sigma^{(0)}$ 已经下降很多个数量级，通常说明搜索分布确实正在收敛。如果 `fmin` 停滞但 `sigma`、`D`、`S` 仍明显变化，算法可能只是在适应搜索分布，而不是已经结束。

## 3. 看 `D`：变量尺度是否严重不一致
`D_i` 越大，说明搜索允许第 $i$ 个坐标有更大标准差，通常意味着目标函数对这个变量相对不敏感。如果 `max(D)/min(D)` 很大，例如达到 $10^6$，说明问题存在严重尺度差异，数值误差风险也会增加。此时应考虑事先重新缩放设计变量。

## 4. 看 `S`：变量之间是否强耦合
如果 `S` 的最大值与最小值相差很大，说明即使消除单变量尺度差异后，仍然存在非常强的相关方向。可以进一步查看 `C` 的特征向量，确定哪些变量组合构成低敏感或高敏感方向。

经验上，在局部二次区域，$DCD$ 经常逐渐接近 Hessian 逆矩阵的形状。这也是为什么 CMA-ES 的日志可以作为一种问题结构诊断工具。

### 结果诊断小结

* **是否充分收敛？** 看 `sigma`，不要只看 `fmin`。
* **变量尺度差异多大？** 看 `D`。
* **变量之间是否存在强依赖？** 看 `S` 和 `C` 的特征向量。
* `D` 或 `S` 的条件数持续发散时，要警惕冗余变量、不可辨识方向或数值精度问题。

CMA-ES 擅长处理病态尺度和变量相关问题，但它需要时间学习协方差。协方差学习阶段目标值暂时下降较慢，并不一定意味着算法无效。

# 优化前应该检查什么

建议至少检查四项：

1. 函数评估预算与问题维数：CMA-ES 是否适合这个问题？
2. 初始搜索分布：$m^{(0)}$、$\sigma^{(0)}$、$D^{(0)}$ 是否合理？
3. 种群规模 $\lambda$：默认值是否符合并行资源和多峰程度？
4. 重启与终止条件：是否会过早停止，或者在已经停滞的搜索上浪费预算？

## 1. 函数评估次数与维数：是否应该使用 CMA-ES

以下场景中，CMA-ES 未必是最合适的首选：

* **维数只有约 5 维或更低**：很多工程问题上 Nelder-Mead 等简单局部搜索可能更省评估。
* **总评估预算连维数的几十倍都不到**：CMA-ES 可能还没来得及把搜索分布收缩，就已经耗尽预算。低维时可以考虑贝叶斯优化；已知有好初值时也可以考虑局部搜索。

但评估预算应按墙钟时间来理解。如果每代 $\lambda$ 个候选解可以并行评估，那么即使函数评估总数较多，实际时间也可能仍然可接受。

## 2. 初始分布参数

默认不知道问题结构时，通常令 $C^{(0)}=I$。如果也不知道各变量相对尺度，可以令 $D^{(0)}=I$。真正需要认真设计的是初始均值和初始尺度。

### 已经有一个可信设计 $x_{guess}$
可以令 $m^{(0)}=x_{guess}$，并使用较小的初始步长做局部搜索。若已知“变量变化 $10^{-4}$ 对性能几乎没有影响”，可以把这个量级用于初始尺度。如果每个变量尺度不同，令 $\sigma^{(0)}=1$，把尺度放入 $D^{(0)}$。CMA-ES 对“步长偏小”通常能较快纠正，因此从一个好解附近做局部精修时，宁可初始步长略保守。

### 只知道变量合理范围 $[L_i,U_i]$
可以随机初始化
$$m_i^{(0)}\sim\mathcal U[L_i,U_i],$$
并例如设置 $\sigma^{(0)}=1$、$D_i^{(0)}=(U_i-L_i)/4$。初始尺度越大越偏向一次较全局的搜索；尺度越小越容易探索不同局部盆地。

实践中往往不是只能跑一次。一个有效策略是：先围绕已有好解以小尺度搜索，然后把尺度逐次放大 10 倍、100 倍，逐渐增加全局性。

### Rosenbrock：初始步长的典型例子

Rosenbrock 函数
$$f(x)=\sum_{i=1}^{d-1}100(x_i^2-x_{i+1})^2+(x_i-1)^2$$
的最优解是 $(1,\dots,1)$，但低值区域沿弯曲谷底分布。

值得比较两种情况：
* 从原点出发，初始步长 $1$ 与 $10^{-3}$：后者虽然过小，但 CSA 可以较快把它增大。
* 从最优解附近出发，初始步长 $10^{-1}$ 与 $1$：若步长过大，搜索会迅速离开这个好初值，失去“局部精修”的意义。

如果初始步长小到触发“步长爆炸/分布发散”类终止条件，通常说明它确实过小，应提高一个数量级后再试。

In [ ]:
def rosenbrock(X):
    X = np.atleast_2d(X)
    return 100.0 * np.sum((X[:, :-1]**2 - X[:, 1:])**2, axis=1) + np.sum((X[:, :-1] - 1.0)**2, axis=1)

### Rastrigin：局部搜索与全局搜索

Rastrigin
$$f(x)=\sum_i x_i^2+10(1-\cos(2\pi x_i))$$
有大量局部最优。若 $m^{(0)}$ 在 $(1,\dots,1)$ 附近，$\sigma^{(0)}=0.1$ 往往只会优化到附近局部最优；较大的初始尺度可能跨越多个局部盆地，得到更好的解，也可能变差。

这说明“增大步长”不是无条件有利，而是改变了搜索的全局性。多峰问题中应结合多次重启比较结果分布，而不是根据一次运行下结论。

## 3. 种群规模 $\lambda$

CMA-ES 的默认超参数已经随维数设计好，但种群规模是最值得主动调整的参数。常见默认量级为
$$\lambda=4+\lfloor3\log d\rfloor.$$

增大种群主要在两种情况下有价值：

1. **你有足够多并行资源**。更大种群允许使用更大的学习率，通常减少所需迭代数，因此在所有候选解都能并行时，可以减少墙钟时间。经验上当 $\lambda$ 大到 $O(d^2)$ 后，继续增加对减少迭代数的收益会明显变小。
2. **目标函数多峰**。更大种群能够覆盖更广区域，提高发现更优局部盆地的概率。

但多峰问题存在反例：更大的种群也可能更稳定地被一个宽阔但次优的谷吸引。

### Double-Sphere：大种群和大步长可能适得其反

考虑
$$f(x)=\min\left[\sum_i(x_i-a_i)^2,\;d+s\sum_i(x_i-b_i)^2\right],\qquad s>0.$$
全局最优谷位于 $a$，另一个局部谷位于 $b$。如果 $s$ 很小，次优谷的吸引区域可能远大于全局最优谷。此时增大初始步长或种群规模，会让采样更容易“只看到”宽谷，反而降低发现窄全局谷的概率。

这就是为什么在真正的全局多峰问题中，不能简单套用“种群越大越好”。

## 4. 重启与终止条件

真实应用中，一次搜索就找到满意解的概率往往不高，因此应该把重启视为算法的一部分，而不是失败后的临时补救。常见策略包括 IPOP、BIPOP；本教程前面的多峰章节已经介绍了 IPOP，也就是每次重启扩大种群。

单次运行必须有可靠终止条件，例如：

* 搜索分布已经缩小到工程上无意义的尺度；
* 协方差条件数过大，数值精度风险显著；
* 多代最佳/中位目标值没有改善；
* 达到目标函数阈值；
* 达到最大函数评估预算。

默认阈值通常从浮点数值精度出发，但工程问题往往有自己的实际精度。例如零件加工精度只有 $10^{-3}$，继续把设计变量优化到 $10^{-10}$ 没有意义。把工程分辨率纳入终止条件，可以直接节约大量评估。

# 如果仍然得不到好解：重新检查问题定式化

当初始分布、种群、重启、终止条件都已经检查过，优化仍然异常，就应考虑目标函数本身是否给 CMA-ES 制造了不必要的困难。实践中尤其常见四类问题：

1. 几乎平坦的多峰/周期函数；
2. 对目标函数没有影响的冗余变量；
3. 等高线存在尖角或零开角；
4. 真正的全局多峰结构。

## 1. 几乎平坦的多峰函数或周期函数

CMA-ES 只使用候选解排名，因此对目标值的单调缩放天然不敏感。例如 Sphere 与它的任意严格单调变换，在 CMA-ES 看来具有相同排序。**小梯度本身不是问题**。

真正危险的是：在大范围内，目标函数几乎只剩周期起伏。例如 Ackley
$$f(x)=20-20\exp\left(-0.2\sqrt{\frac1d\sum_i x_i^2}\right)+e-\exp\left(\frac1d\sum_i\cos(2\pi x_i)\right).$$
在合理范围内它仍有明显指向原点的宏观结构；如果搜索范围无限扩大，远处第一项近似饱和，算法看到的几乎只是周期项，搜索可能变得不稳定，$\sigma D$ 甚至异常增大。

**对策**：如果物理上知道合理设计范围，就明确加入边界；不要让算法探索毫无意义的极远区域。

In [ ]:
def ackley(X):
    X = np.atleast_2d(X)
    f1 = 20.0 * (1.0 - np.exp(-0.2 * np.sqrt(np.mean(X**2, axis=1))))
    f2 = np.e - np.exp(np.mean(np.cos(2.0*np.pi*X), axis=1))
    return f1 + f2

## 2. 对目标函数没有影响的冗余变量

设一个 100 维变量中只有前 20 或 50 维真正影响目标函数，其余维度完全无关。虽然“有效问题”本质上仍只是低维 Sphere，但 CMA-ES 不知道哪些方向无关。结果通常是：

* $\sigma$ 收缩更慢，需要更多评估；
* 无关方向的方差会相对不断变大，协方差条件数可能持续发散；
* 高精度阶段容易出现数值问题。

如果冗余维度比例不高、当前精度已经满足需求，可以不处理。但如果条件数持续发散并阻碍高精度优化，应考虑消除无效变量或根据搜索得到的协方差做降维。

In [ ]:
# 假设 Cov 是一次 CMA-ES 搜索后得到的协方差矩阵
# eigval, eigvec = np.linalg.eigh(Cov)
# 选择较小特征值对应、真正影响目标函数的子空间作为新搜索基底
# basis = eigvec[:, eigval < threshold]
# x = x_base + z @ basis.T

def reconstruct(z, x_base, basis):
    return x_base + np.dot(z, basis.T)

## 3. 尖角等高线

不可微本身并不会自动让 CMA-ES 失败，因为它不使用梯度。但如果等高线形成非常尖的角，尤其开角趋近 0，就可能出现提前收敛到非最优尖点的现象。一个典型族是
$$f(x)=\left(\sum_i|x_i|^p\right)^{1/p}.$$
当 $p$ 很小，例如 $p=1/4$，等高线会变得非常尖。默认小种群可能过早收缩；增大种群有时能够缓解，但从源头修改目标函数通常更好。

### 对策：避免人为制造尖角
尖角经常来自 `max`、`min`、硬条件分支或惩罚函数。例如把约束直接写成 $f(x)+c\max(g(x)-G,0)$，会在约束边界引入不光滑结构。

可以考虑用平滑近似，例如 LogSumExp：
$$\max(z_1,\dots,z_k)\approx\frac1a\log\left(\sum_i e^{az_i}\right),\qquad a>0.$$
为数值稳定，常写成
$$z_{max}+\frac1a\log\left(\sum_i e^{a(z_i-z_{max})}\right).$$

并不是所有非光滑问题都要平滑；关键是检查最优解是否恰好位于尖角/约束边界，以及这种尖角是否让搜索分布出现异常收缩。

## 4. 真正的全局多峰问题

对于 Double-Sphere 这类全局多峰问题，增大种群和增大步长可能都无效甚至反效果，最终只能依靠不同初值、不同尺度和重启进行覆盖。

如果存在额外领域知识，可以设计一个辅助函数 $g(x)$：已知真正希望的解应当让 $g(x)$ 也较小，那么先优化
$$f(x)+c g(x),\qquad c>0,$$
把搜索引导到更合理的全局盆地，再以得到的解作为初值、重新只优化原始 $f(x)$ 做局部精修。

需要强调：辅助目标只是**引导**。加了 $g$ 后得到的解一般不再是原始 $f$ 的局部最优，因此最后应该回到原始问题进行精修。

# 其他重要主题

## 约束处理
不同约束能提供的信息不同，不存在一种通用最佳处理法。Box constraint 通常可以用镜像、截断或变量变换；一般非线性约束常使用惩罚或可行性规则，但惩罚系数太大可能制造病态尺度和尖角，太小又可能让最终解违反约束。应根据约束结构选择方法。

## 仿真条件不确定性
如果目标函数来自仿真，而天气、路面、载荷、模型参数等条件在真实部署中不确定，那么固定一个仿真条件优化出来的解可能并不鲁棒。可以考虑最坏情况、风险度量或 Min-Max 优化。本仓库 `a1_minmax_optimization.ipynb` 专门介绍 Adversarial-CMA-ES 与 WRA-CMA-ES。

## 仿真太慢
若昂贵仿真导致预算太少，CMA-ES 可能无法发挥优势。除了并行候选解评估，还可以考虑降低单次仿真精度，再通过 multi-fidelity optimization 在搜索过程中动态控制精度。不要简单地全程使用低精度仿真，否则优化可能只是在拟合低保真模型误差。

执行时间的进一步讨论见 `cmaes_acceleration.ipynb`。

# 一页式实践建议

当 CMA-ES “效果不好”时，建议按以下顺序处理：

1. 先看是否有足够函数评估预算。
2. 看 `sigma`，确认搜索是收敛、停滞还是仍在适应。
3. 看 `D` 与 `S`，判断尺度差异和变量耦合。
4. 检查初始均值和初始尺度是否符合你真正想做的“局部/全局”搜索。
5. 多峰问题不要只跑一次；使用重启，并主动改变种群与初始尺度。
6. 若协方差条件数持续发散，检查冗余变量和不可辨识方向。
7. 若搜索在边界或尖点异常，检查 `max/min`、硬分支和惩罚函数是否制造了不必要的尖角。
8. 如果问题本身是全局多峰，算法调参不能替代领域知识；必要时加入合理引导目标，再回到原目标精修。

CMA-ES 的价值不仅是“自动找一个最优解”，它学习出的分布参数本身也可以帮助理解目标函数的局部几何结构。

# 结语

实际应用中的困难远多于一个教程能够覆盖的范围。更有效的使用方式，是把 CMA-ES 当作一个可诊断的搜索过程：既观察目标值，也观察分布如何演化，再据此判断问题究竟来自算法配置、计算预算、变量表示、约束处理，还是目标函数本身。